# Predicting Environmental Compliance Violations at Small Manufacturing Facilities
## ISYE 6740 Final Project — Team 157
**Pramodh Aryasomayajula & Keshav Shah**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import re

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report, f1_score, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance
import xgboost as xgb
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import shap

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

RANDOM_STATE = 42
FIGURES_DIR = 'figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

print('All imports successful.')

---
## 1. Data Loading & Filtering

In [ ]:
cols_to_load = [
    'REGISTRY_ID', 'FAC_NAME', 'FAC_STATE', 'FAC_EPA_REGION',
    'FAC_LAT', 'FAC_LONG', 'FAC_COUNTY',
    'FAC_NAICS_CODES', 'FAC_MAJOR_FLAG', 'FAC_ACTIVE_FLAG',
    'FAC_PERCENT_MINORITY', 'FAC_POP_DEN',
    'AIR_FLAG', 'NPDES_FLAG', 'SDWIS_FLAG', 'RCRA_FLAG', 'TRI_FLAG', 'GHG_FLAG',
    'FAC_INSPECTION_COUNT', 'FAC_INFORMAL_COUNT', 'FAC_FORMAL_ACTION_COUNT',
    'FAC_TOTAL_PENALTIES', 'FAC_PENALTY_COUNT',
    'FAC_3YR_COMPLIANCE_HISTORY', 'FAC_QTRS_WITH_NC',
    'FAC_COMPLIANCE_STATUS',
    'CAA_EVALUATION_COUNT', 'CWA_INSPECTION_COUNT', 'RCRA_INSPECTION_COUNT',
    'CAA_3YR_COMPL_QTRS_HISTORY', 'CWA_13QTRS_COMPL_HISTORY',
    'RCRA_3YR_COMPL_QTRS_HISTORY'
]

print('Loading ECHO_EXPORTER.csv (selected columns only)...')
chunks = []
for chunk in pd.read_csv('ECHO_EXPORTER.csv', usecols=cols_to_load,
                         dtype=str, chunksize=200_000, low_memory=False):
    mask = chunk['FAC_NAICS_CODES'].fillna('').str.contains(r'\b3[123]', regex=True)
    chunks.append(chunk[mask])

df_raw = pd.concat(chunks, ignore_index=True)
print(f'After NAICS 31-33 filter: {len(df_raw):,} facilities')

In [ ]:
df = df_raw.copy()
df = df[df['FAC_MAJOR_FLAG'] != 'Y']
print(f'After excluding major facilities: {len(df):,}')

df = df[df['FAC_ACTIVE_FLAG'] == 'Y']
print(f'After keeping only active facilities: {len(df):,}')

df = df[df['FAC_3YR_COMPLIANCE_HISTORY'].fillna('').str.len() == 12]
print(f'After requiring 12-quarter compliance history: {len(df):,}')

df = df.reset_index(drop=True)
print(f'\nFinal filtered dataset: {len(df):,} facilities')

---
## 2. Target Variable & Feature Engineering

In [ ]:
def parse_history_features(history_str, feature_end=8):
    """Parse a compliance history string into features using only the feature window."""
    feat_window = history_str[:feature_end]
    n_v = feat_window.count('V')
    n_s = feat_window.count('S')
    n_nc = n_v + n_s
    n_unknown = feat_window.count('U')
    n_clean = sum(1 for c in feat_window if c in ('_', ' '))

    last_nc_dist = feature_end
    for i in range(feature_end - 1, -1, -1):
        if feat_window[i] in ('V', 'S'):
            last_nc_dist = (feature_end - 1) - i
            break

    first_half_nc = sum(1 for c in feat_window[:feature_end // 2] if c in ('V', 'S'))
    second_half_nc = sum(1 for c in feat_window[feature_end // 2:] if c in ('V', 'S'))
    trend = second_half_nc - first_half_nc

    max_streak = 0
    current_streak = 0
    for c in feat_window:
        if c in ('_', ' '):
            current_streak += 1
            max_streak = max(max_streak, current_streak)
        else:
            current_streak = 0

    weights = np.array([2 ** (i / (feature_end - 1)) for i in range(feature_end)])
    weighted = sum(weights[i] for i in range(feature_end) if feat_window[i] in ('V', 'S'))

    return {
        'n_violation_qtrs': n_v,
        'n_snc_qtrs': n_s,
        'n_nc_qtrs': n_nc,
        'n_unknown_qtrs': n_unknown,
        'n_clean_qtrs': n_clean,
        'last_nc_distance': last_nc_dist,
        'violation_trend': trend,
        'longest_clean_streak': max_streak,
        'violation_recency_weighted': weighted,
    }

hist = df['FAC_3YR_COMPLIANCE_HISTORY']

# Target: any V or S in positions 8-11 (most recent 4 quarters)
df['target'] = hist.apply(lambda h: int(any(c in ('V', 'S') for c in h[8:12])))
print(f'Target distribution:\n{df["target"].value_counts()}\n')
print(f'Positive class rate: {df["target"].mean():.3%}')

In [ ]:
# Parse general compliance history (positions 0-7)
hist_features = hist.apply(lambda h: parse_history_features(h, 8))
hist_df = pd.DataFrame(hist_features.tolist())
hist_df.columns = ['fac_' + c for c in hist_df.columns]

# Parse CAA history (12 chars, positions 0-7)
caa_hist = df['CAA_3YR_COMPL_QTRS_HISTORY'].fillna('')
has_caa = caa_hist.str.len() >= 12
caa_features = caa_hist[has_caa].apply(lambda h: parse_history_features(h, 8))
caa_df = pd.DataFrame(index=df.index, columns=[f'caa_{c}' for c in ['n_nc_qtrs', 'n_snc_qtrs', 'violation_trend']])
if len(caa_features) > 0:
    caa_parsed = pd.DataFrame(caa_features.tolist(), index=caa_features.index)
    caa_df.loc[has_caa, 'caa_n_nc_qtrs'] = caa_parsed['n_nc_qtrs'].values
    caa_df.loc[has_caa, 'caa_n_snc_qtrs'] = caa_parsed['n_snc_qtrs'].values
    caa_df.loc[has_caa, 'caa_violation_trend'] = caa_parsed['violation_trend'].values
caa_df = caa_df.fillna(0).astype(float)

# Parse RCRA history (12 chars, positions 0-7)
rcra_hist = df['RCRA_3YR_COMPL_QTRS_HISTORY'].fillna('')
has_rcra = rcra_hist.str.len() >= 12
rcra_features = rcra_hist[has_rcra].apply(lambda h: parse_history_features(h, 8))
rcra_df = pd.DataFrame(index=df.index, columns=[f'rcra_{c}' for c in ['n_nc_qtrs', 'n_snc_qtrs', 'violation_trend']])
if len(rcra_features) > 0:
    rcra_parsed = pd.DataFrame(rcra_features.tolist(), index=rcra_features.index)
    rcra_df.loc[has_rcra, 'rcra_n_nc_qtrs'] = rcra_parsed['n_nc_qtrs'].values
    rcra_df.loc[has_rcra, 'rcra_n_snc_qtrs'] = rcra_parsed['n_snc_qtrs'].values
    rcra_df.loc[has_rcra, 'rcra_violation_trend'] = rcra_parsed['violation_trend'].values
rcra_df = rcra_df.fillna(0).astype(float)

# Parse CWA history (13 chars, positions 0-8 as features)
cwa_hist = df['CWA_13QTRS_COMPL_HISTORY'].fillna('')
has_cwa = cwa_hist.str.len() >= 13
cwa_features = cwa_hist[has_cwa].apply(lambda h: parse_history_features(h, 9))
cwa_df = pd.DataFrame(index=df.index, columns=[f'cwa_{c}' for c in ['n_nc_qtrs', 'n_snc_qtrs', 'violation_trend']])
if len(cwa_features) > 0:
    cwa_parsed = pd.DataFrame(cwa_features.tolist(), index=cwa_features.index)
    cwa_df.loc[has_cwa, 'cwa_n_nc_qtrs'] = cwa_parsed['n_nc_qtrs'].values
    cwa_df.loc[has_cwa, 'cwa_n_snc_qtrs'] = cwa_parsed['n_snc_qtrs'].values
    cwa_df.loc[has_cwa, 'cwa_violation_trend'] = cwa_parsed['violation_trend'].values
cwa_df = cwa_df.fillna(0).astype(float)

print('Compliance history features parsed.')
print(f'  CAA history available for {has_caa.sum():,} facilities')
print(f'  CWA history available for {has_cwa.sum():,} facilities')
print(f'  RCRA history available for {has_rcra.sum():,} facilities')

In [ ]:
# Extract primary NAICS code
def extract_primary_mfg_naics(naics_str):
    if pd.isna(naics_str):
        return None
    codes = re.findall(r'\d+', str(naics_str))
    for code in codes:
        if code[:2] in ('31', '32', '33'):
            return code
    return codes[0] if codes else None

df['primary_naics'] = df['FAC_NAICS_CODES'].apply(extract_primary_mfg_naics)
df['naics_2digit'] = df['primary_naics'].str[:2]
df['naics_3digit'] = df['primary_naics'].str[:3]

# Program flags to binary
program_flags = ['AIR_FLAG', 'NPDES_FLAG', 'SDWIS_FLAG', 'RCRA_FLAG', 'TRI_FLAG', 'GHG_FLAG']
for col in program_flags:
    df[col.lower()] = (df[col] == 'Y').astype(int)

df['n_programs'] = df[[c.lower() for c in program_flags]].sum(axis=1)

# Numeric columns
numeric_cols_raw = ['FAC_INSPECTION_COUNT', 'FAC_INFORMAL_COUNT', 'FAC_FORMAL_ACTION_COUNT',
                    'FAC_TOTAL_PENALTIES', 'FAC_PENALTY_COUNT',
                    'CAA_EVALUATION_COUNT', 'CWA_INSPECTION_COUNT', 'RCRA_INSPECTION_COUNT',
                    'FAC_PERCENT_MINORITY', 'FAC_POP_DEN', 'FAC_LAT', 'FAC_LONG']
for col in numeric_cols_raw:
    df[col.lower()] = pd.to_numeric(df[col], errors='coerce')

# Program-specific history availability indicators
df['has_caa_history'] = has_caa.astype(int)
df['has_cwa_history'] = has_cwa.astype(int)
df['has_rcra_history'] = has_rcra.astype(int)

# EPA region
df['fac_epa_region'] = df['FAC_EPA_REGION'].fillna('Unknown')

print(f'NAICS 2-digit distribution:\n{df["naics_2digit"].value_counts()}')
print(f'\nPrograms per facility:\n{df["n_programs"].value_counts().sort_index()}')

In [ ]:
# Assemble the final feature matrix
feature_cols_numeric = [
    # General compliance history (from feature window)
    'fac_n_violation_qtrs', 'fac_n_snc_qtrs', 'fac_n_nc_qtrs',
    'fac_n_unknown_qtrs', 'fac_n_clean_qtrs',
    'fac_last_nc_distance', 'fac_violation_trend',
    'fac_longest_clean_streak', 'fac_violation_recency_weighted',
    # Program-specific history
    'caa_n_nc_qtrs', 'caa_n_snc_qtrs', 'caa_violation_trend',
    'cwa_n_nc_qtrs', 'cwa_n_snc_qtrs', 'cwa_violation_trend',
    'rcra_n_nc_qtrs', 'rcra_n_snc_qtrs', 'rcra_violation_trend',
    # Regulatory activity (cumulative — noted as potential soft leakage)
    'fac_inspection_count', 'fac_informal_count', 'fac_formal_action_count',
    'fac_total_penalties', 'fac_penalty_count',
    'caa_evaluation_count', 'cwa_inspection_count', 'rcra_inspection_count',
    # Demographics and geography
    'fac_percent_minority', 'fac_pop_den', 'fac_lat', 'fac_long',
    # Program count and availability
    'n_programs', 'has_caa_history', 'has_cwa_history', 'has_rcra_history',
]

feature_cols_binary = ['air_flag', 'npdes_flag', 'sdwis_flag', 'rcra_flag', 'tri_flag', 'ghg_flag']
feature_cols_te = ['naics_3digit', 'fac_state']
feature_cols_ohe = ['fac_epa_region', 'naics_2digit']

# Build feature dataframe
features = pd.concat([
    hist_df,
    caa_df,
    cwa_df,
    rcra_df,
    df[['fac_inspection_count', 'fac_informal_count', 'fac_formal_action_count',
        'fac_total_penalties', 'fac_penalty_count',
        'caa_evaluation_count', 'cwa_inspection_count', 'rcra_inspection_count',
        'fac_percent_minority', 'fac_pop_den', 'fac_lat', 'fac_long',
        'n_programs', 'has_caa_history', 'has_cwa_history', 'has_rcra_history',
        'air_flag', 'npdes_flag', 'sdwis_flag', 'rcra_flag', 'tri_flag', 'ghg_flag']],
    df[['naics_3digit', 'naics_2digit']].rename(columns={'naics_3digit': 'naics_3digit', 'naics_2digit': 'naics_2digit'}),
    df[['FAC_STATE']].rename(columns={'FAC_STATE': 'fac_state'}),
    df[['fac_epa_region']],
], axis=1)

# Fill NaN in numeric features
for col in feature_cols_numeric:
    if col in features.columns:
        features[col] = pd.to_numeric(features[col], errors='coerce')

# Fill categorical NaN
features['naics_3digit'] = features['naics_3digit'].fillna('Unknown')
features['naics_2digit'] = features['naics_2digit'].fillna('Unknown')
features['fac_state'] = features['fac_state'].fillna('Unknown')
features['fac_epa_region'] = features['fac_epa_region'].fillna('Unknown')

y = df['target'].values

print(f'Feature matrix shape: {features.shape}')
print(f'Target shape: {y.shape}')
print(f'\nMissingness in numeric features:')
missing = features[feature_cols_numeric].isnull().mean()
print(missing[missing > 0].sort_values(ascending=False))

---
## 3. Exploratory Data Analysis

In [ ]:
# 3.1 Target class distribution
fig, ax = plt.subplots(figsize=(6, 4))
counts = df['target'].value_counts().sort_index()
bars = ax.bar(['Compliant (0)', 'Violation (1)'], counts.values,
              color=['#2ecc71', '#e74c3c'], edgecolor='black')
for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f'{count:,}\n({count/len(df):.1%})', ha='center', fontsize=11)
ax.set_ylabel('Number of Facilities')
ax.set_title('Target Variable Distribution')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/target_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# 3.2 Non-compliance quarters by target class
fig, ax = plt.subplots(figsize=(8, 5))
for label, color, name in [(0, '#2ecc71', 'Compliant'), (1, '#e74c3c', 'Violation')]:
    subset = hist_df.loc[df['target'] == label, 'fac_n_nc_qtrs']
    ax.hist(subset, bins=range(0, 10), alpha=0.6, color=color, label=name, edgecolor='black')
ax.set_xlabel('Non-Compliant Quarters in Feature Window (8 quarters)')
ax.set_ylabel('Number of Facilities')
ax.set_title('Historical Non-Compliance by Target Class')
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/nc_quarters_by_class.png', bbox_inches='tight')
plt.show()

In [ ]:
# 3.3 Violation rate by NAICS 3-digit subsector (top 15)
naics_stats = df.groupby('naics_3digit').agg(
    count=('target', 'size'),
    violation_rate=('target', 'mean')
).reset_index()
naics_top15 = naics_stats.nlargest(15, 'count').sort_values('violation_rate', ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.barh(naics_top15['naics_3digit'], naics_top15['violation_rate'],
               color='#3498db', edgecolor='black')
for bar, count in zip(bars, naics_top15['count']):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'n={count:,}', va='center', fontsize=9)
ax.set_xlabel('Violation Rate')
ax.set_ylabel('NAICS 3-Digit Subsector')
ax.set_title('Violation Rate by NAICS Subsector (Top 15 by Count)')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/violation_rate_naics.png', bbox_inches='tight')
plt.show()

In [ ]:
# 3.4 Violation rate by state (top 20)
state_stats = df.groupby('FAC_STATE').agg(
    count=('target', 'size'),
    violation_rate=('target', 'mean')
).reset_index()
state_top20 = state_stats.nlargest(20, 'count').sort_values('violation_rate', ascending=True)

fig, ax = plt.subplots(figsize=(8, 7))
bars = ax.barh(state_top20['FAC_STATE'], state_top20['violation_rate'],
               color='#9b59b6', edgecolor='black')
for bar, count in zip(bars, state_top20['count']):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f'n={count:,}', va='center', fontsize=9)
ax.set_xlabel('Violation Rate')
ax.set_ylabel('State')
ax.set_title('Violation Rate by State (Top 20 by Facility Count)')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/violation_rate_state.png', bbox_inches='tight')
plt.show()

In [ ]:
# 3.5 Inspection count by target class
fig, ax = plt.subplots(figsize=(8, 5))
plot_data = pd.DataFrame({
    'Inspections': df['fac_inspection_count'].clip(upper=20),
    'Class': df['target'].map({0: 'Compliant', 1: 'Violation'})
})
sns.violinplot(data=plot_data, x='Class', y='Inspections', palette=['#2ecc71', '#e74c3c'],
               inner='quartile', ax=ax)
ax.set_ylabel('Inspection Count (capped at 20)')
ax.set_title('Inspection Count Distribution by Target Class')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/inspection_violin.png', bbox_inches='tight')
plt.show()

In [ ]:
# 3.6 Correlation heatmap of numeric features
corr_cols = [
    'fac_n_nc_qtrs', 'fac_n_snc_qtrs', 'fac_violation_trend',
    'fac_last_nc_distance', 'fac_longest_clean_streak',
    'fac_violation_recency_weighted',
    'fac_inspection_count', 'fac_informal_count', 'fac_formal_action_count',
    'fac_total_penalties', 'n_programs',
    'fac_percent_minority', 'fac_pop_den'
]
corr_data = features[corr_cols].apply(pd.to_numeric, errors='coerce')
corr_matrix = corr_data.corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/correlation_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# 3.7 Violation rate by number of programs
prog_stats = df.groupby('n_programs').agg(
    count=('target', 'size'),
    violation_rate=('target', 'mean')
).reset_index()

fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.bar(prog_stats['n_programs'], prog_stats['violation_rate'],
        color='#e74c3c', alpha=0.7, edgecolor='black', label='Violation Rate')
ax1.set_xlabel('Number of Active Regulatory Programs')
ax1.set_ylabel('Violation Rate', color='#e74c3c')
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

ax2 = ax1.twinx()
ax2.plot(prog_stats['n_programs'], prog_stats['count'], 'o-',
         color='#2c3e50', linewidth=2, label='Facility Count')
ax2.set_ylabel('Number of Facilities', color='#2c3e50')

ax1.set_title('Violation Rate and Facility Count by Number of Programs')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/violation_rate_programs.png', bbox_inches='tight')
plt.show()

In [ ]:
# 3.8 Missingness summary
all_feature_cols = feature_cols_numeric + feature_cols_binary + feature_cols_te + feature_cols_ohe
missing_summary = pd.DataFrame({
    'Feature': all_feature_cols,
    'Missing Count': [features[c].isnull().sum() if c in features.columns else 0 for c in all_feature_cols],
    'Missing %': [features[c].isnull().mean() * 100 if c in features.columns else 0 for c in all_feature_cols]
})
missing_summary = missing_summary[missing_summary['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print('Features with missing values:')
print(missing_summary.to_string(index=False))
if len(missing_summary) == 0:
    print('No missing values after preprocessing.')

---
## 4. Preprocessing & Train/Test Split

In [ ]:
# Impute remaining NaN in numeric features
for col in feature_cols_numeric:
    if col in features.columns and features[col].isnull().any():
        features[col] = features[col].fillna(features[col].median())

X = features[feature_cols_numeric + feature_cols_binary + feature_cols_te + feature_cols_ohe].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Training set: {X_train.shape[0]:,} samples ({y_train.mean():.2%} positive)')
print(f'Test set:     {X_test.shape[0]:,} samples ({y_test.mean():.2%} positive)')

In [ ]:
# Build preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), feature_cols_numeric),
        ('bin', 'passthrough', feature_cols_binary),
        ('te', TargetEncoder(smooth='auto', random_state=RANDOM_STATE), feature_cols_te),
        ('ohe', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='infrequent_if_exist'),
         feature_cols_ohe),
    ],
    remainder='drop'
)

print('Preprocessing pipeline defined.')
print(f'  Numeric features: {len(feature_cols_numeric)}')
print(f'  Binary features: {len(feature_cols_binary)}')
print(f'  Target-encoded features: {len(feature_cols_te)}')
print(f'  One-hot-encoded features: {len(feature_cols_ohe)}')

---
## 5. Model Training

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = {}

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'Class ratio (neg/pos): {scale_pos_weight:.1f}')

In [ ]:
# 5.1 Logistic Regression (L1)
print('Training Logistic Regression (L1)...')
lr_pipe = Pipeline([
    ('pre', preprocessor),
    ('clf', LogisticRegression(penalty='l1', solver='saga', class_weight='balanced',
                               max_iter=5000, random_state=RANDOM_STATE))
])

lr_params = {'clf__C': [0.001, 0.01, 0.1, 1, 10]}
lr_search = GridSearchCV(lr_pipe, lr_params, cv=cv, scoring='roc_auc', n_jobs=-1, refit=True)
lr_search.fit(X_train, y_train)

results['Logistic Regression'] = {
    'model': lr_search.best_estimator_,
    'best_params': lr_search.best_params_,
    'cv_auc': lr_search.best_score_,
    'y_prob': lr_search.best_estimator_.predict_proba(X_test)[:, 1]
}
print(f'  Best C: {lr_search.best_params_["clf__C"]}')
print(f'  CV AUC-ROC: {lr_search.best_score_:.4f}')
print(f'  Test AUC-ROC: {roc_auc_score(y_test, results["Logistic Regression"]["y_prob"]):.4f}')

In [ ]:
# 5.2 Random Forest
print('Training Random Forest...')
rf_pipe = Pipeline([
    ('pre', preprocessor),
    ('clf', RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1))
])

rf_params = {
    'clf__n_estimators': [200, 500],
    'clf__max_depth': [5, 10, 20],
    'clf__min_samples_leaf': [5, 20, 50]
}
rf_search = GridSearchCV(rf_pipe, rf_params, cv=cv, scoring='roc_auc', n_jobs=-1, refit=True)
rf_search.fit(X_train, y_train)

results['Random Forest'] = {
    'model': rf_search.best_estimator_,
    'best_params': rf_search.best_params_,
    'cv_auc': rf_search.best_score_,
    'y_prob': rf_search.best_estimator_.predict_proba(X_test)[:, 1]
}
print(f'  Best params: {rf_search.best_params_}')
print(f'  CV AUC-ROC: {rf_search.best_score_:.4f}')
print(f'  Test AUC-ROC: {roc_auc_score(y_test, results["Random Forest"]["y_prob"]):.4f}')

In [ ]:
# 5.3 XGBoost
print('Training XGBoost...')
xgb_pipe = Pipeline([
    ('pre', preprocessor),
    ('clf', xgb.XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        eval_metric='auc',
        tree_method='hist',
        random_state=RANDOM_STATE,
        use_label_encoder=False
    ))
])

xgb_params = {
    'clf__max_depth': [3, 5, 7],
    'clf__learning_rate': [0.01, 0.05, 0.1],
    'clf__n_estimators': [200, 500, 1000],
    'clf__subsample': [0.7, 0.9],
    'clf__colsample_bytree': [0.7, 0.9],
}
xgb_search = RandomizedSearchCV(
    xgb_pipe, xgb_params, n_iter=30, cv=cv, scoring='roc_auc',
    n_jobs=-1, refit=True, random_state=RANDOM_STATE
)
xgb_search.fit(X_train, y_train)

results['XGBoost'] = {
    'model': xgb_search.best_estimator_,
    'best_params': xgb_search.best_params_,
    'cv_auc': xgb_search.best_score_,
    'y_prob': xgb_search.best_estimator_.predict_proba(X_test)[:, 1]
}
print(f'  Best params: {xgb_search.best_params_}')
print(f'  CV AUC-ROC: {xgb_search.best_score_:.4f}')
print(f'  Test AUC-ROC: {roc_auc_score(y_test, results["XGBoost"]["y_prob"]):.4f}')

In [ ]:
# 5.4 SVM (RBF) — trained on subsample due to O(n^2) scaling
print('Training SVM (RBF kernel) on subsample...')
SVM_SAMPLE_SIZE = 12000

np.random.seed(RANDOM_STATE)
svm_idx = np.random.choice(len(X_train), size=min(SVM_SAMPLE_SIZE, len(X_train)), replace=False)
X_train_svm = X_train.iloc[svm_idx]
y_train_svm = y_train[svm_idx]
print(f'  SVM subsample: {len(X_train_svm):,} rows ({y_train_svm.mean():.2%} positive)')

svm_pipe = Pipeline([
    ('pre', preprocessor),
    ('clf', SVC(kernel='rbf', class_weight='balanced', probability=True,
                random_state=RANDOM_STATE))
])

svm_params = {'clf__C': [0.1, 1, 10], 'clf__gamma': ['scale', 0.01, 0.001]}
svm_search = GridSearchCV(svm_pipe, svm_params, cv=cv, scoring='roc_auc', n_jobs=-1, refit=True)
svm_search.fit(X_train_svm, y_train_svm)

results['SVM (RBF)'] = {
    'model': svm_search.best_estimator_,
    'best_params': svm_search.best_params_,
    'cv_auc': svm_search.best_score_,
    'y_prob': svm_search.best_estimator_.predict_proba(X_test)[:, 1]
}
print(f'  Best params: {svm_search.best_params_}')
print(f'  CV AUC-ROC: {svm_search.best_score_:.4f}')
print(f'  Test AUC-ROC: {roc_auc_score(y_test, results["SVM (RBF)"]["y_prob"]):.4f}')

---
## 6. Class Imbalance Experiments (XGBoost)

In [ ]:
best_xgb_params = {k.replace('clf__', ''): v for k, v in xgb_search.best_params_.items()}

imbalance_results = {}

# 6.1 No class weighting
print('Imbalance experiment: no weighting...')
xgb_no_weight = Pipeline([
    ('pre', preprocessor),
    ('clf', xgb.XGBClassifier(**best_xgb_params, eval_metric='auc',
                              tree_method='hist', random_state=RANDOM_STATE))
])
xgb_no_weight.fit(X_train, y_train)
prob_no_weight = xgb_no_weight.predict_proba(X_test)[:, 1]
imbalance_results['No Weighting'] = {
    'auc_roc': roc_auc_score(y_test, prob_no_weight),
    'auc_pr': average_precision_score(y_test, prob_no_weight)
}

# 6.2 Class weighting (already done — the main XGBoost result)
prob_weighted = results['XGBoost']['y_prob']
imbalance_results['Class Weighting'] = {
    'auc_roc': roc_auc_score(y_test, prob_weighted),
    'auc_pr': average_precision_score(y_test, prob_weighted)
}

# 6.3 SMOTE
print('Imbalance experiment: SMOTE...')
xgb_smote = ImbPipeline([
    ('pre', preprocessor),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('clf', xgb.XGBClassifier(**best_xgb_params, eval_metric='auc',
                              tree_method='hist', random_state=RANDOM_STATE))
])
xgb_smote.fit(X_train, y_train)
prob_smote = xgb_smote.predict_proba(X_test)[:, 1]
imbalance_results['SMOTE'] = {
    'auc_roc': roc_auc_score(y_test, prob_smote),
    'auc_pr': average_precision_score(y_test, prob_smote)
}

imb_df = pd.DataFrame(imbalance_results).T
imb_df.index.name = 'Strategy'
print('\nClass Imbalance Experiment Results:')
print(imb_df.round(4).to_string())

---
## 7. Evaluation

In [ ]:
# 7.1 Bootstrap confidence intervals
def bootstrap_auc(y_true, y_scores, n_bootstraps=1000, seed=42):
    rng = np.random.RandomState(seed)
    aucs = []
    for _ in range(n_bootstraps):
        indices = rng.randint(0, len(y_true), len(y_true))
        if len(np.unique(y_true[indices])) < 2:
            continue
        aucs.append(roc_auc_score(y_true[indices], y_scores[indices]))
    lower, upper = np.percentile(aucs, [2.5, 97.5])
    return np.mean(aucs), lower, upper

print('Computing bootstrap CIs (1000 resamples)...')
model_metrics = {}
for name, res in results.items():
    y_prob = res['y_prob']
    auc_mean, auc_lo, auc_hi = bootstrap_auc(y_test, y_prob)
    auc_pr = average_precision_score(y_test, y_prob)

    fpr, tpr, thresholds = roc_curve(y_test, y_prob)
    j_scores = tpr - fpr
    best_thresh = thresholds[np.argmax(j_scores)]
    y_pred = (y_prob >= best_thresh).astype(int)

    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    model_metrics[name] = {
        'AUC-ROC': auc_mean, 'AUC-ROC CI': f'[{auc_lo:.4f}, {auc_hi:.4f}]',
        'AUC-PR': auc_pr,
        'Threshold': best_thresh,
        'Precision': precision, 'Recall': recall, 'F1': f1,
        'y_pred': y_pred, 'cm': cm
    }
    print(f'{name}: AUC-ROC={auc_mean:.4f} [{auc_lo:.4f}, {auc_hi:.4f}], '
          f'AUC-PR={auc_pr:.4f}, F1={f1:.4f}')

# Summary table
summary_df = pd.DataFrame({
    name: {k: v for k, v in m.items() if k not in ('y_pred', 'cm')}
    for name, m in model_metrics.items()
}).T
print('\n--- Model Comparison Summary ---')
print(summary_df.to_string())

In [ ]:
# 7.2 ROC Curves
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']
for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    auc = roc_auc_score(y_test, res['y_prob'])
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC={auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/roc_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
# 7.3 Precision-Recall Curves
fig, ax = plt.subplots(figsize=(8, 6))
for (name, res), color in zip(results.items(), colors):
    prec, rec, _ = precision_recall_curve(y_test, res['y_prob'])
    ap = average_precision_score(y_test, res['y_prob'])
    ax.plot(rec, prec, color=color, linewidth=2, label=f'{name} (AP={ap:.3f})')
baseline = y_test.mean()
ax.axhline(y=baseline, color='k', linestyle='--', linewidth=1, alpha=0.5, label=f'Baseline ({baseline:.3f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/pr_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
# 7.4 Confusion Matrices
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, m) in zip(axes, model_metrics.items()):
    ConfusionMatrixDisplay(m['cm'], display_labels=['Compliant', 'Violation']).plot(ax=ax, cmap='Blues')
    ax.set_title(name, fontsize=10)
plt.suptitle('Confusion Matrices (Optimal Threshold)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/confusion_matrices.png', bbox_inches='tight')
plt.show()

In [ ]:
# 7.5 AUC-ROC with bootstrap CIs
fig, ax = plt.subplots(figsize=(8, 5))
names = list(model_metrics.keys())
aucs = [model_metrics[n]['AUC-ROC'] for n in names]
ci_strs = [model_metrics[n]['AUC-ROC CI'] for n in names]
ci_los = [float(s.split(',')[0].strip('[')) for s in ci_strs]
ci_his = [float(s.split(',')[1].strip(' ]')) for s in ci_strs]
yerr = [[a - lo for a, lo in zip(aucs, ci_los)],
        [hi - a for a, hi in zip(aucs, ci_his)]]

bars = ax.bar(names, aucs, color=colors[:len(names)], edgecolor='black', alpha=0.8)
ax.errorbar(names, aucs, yerr=yerr, fmt='none', ecolor='black', capsize=5, linewidth=2)
for bar, auc in zip(bars, aucs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{auc:.3f}', ha='center', fontsize=11)
ax.set_ylabel('AUC-ROC')
ax.set_title('Model Comparison: AUC-ROC with 95% Bootstrap CI')
ax.set_ylim(0.5, 1.0)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/auc_comparison.png', bbox_inches='tight')
plt.show()

---
## 8. Feature Importance Analysis

In [ ]:
# Get feature names from the preprocessor
preprocessor_fitted = results['XGBoost']['model'].named_steps['pre']
num_names = feature_cols_numeric
bin_names = feature_cols_binary
te_names = [f'{c}_te' for c in feature_cols_te]
try:
    ohe_names = list(preprocessor_fitted.named_transformers_['ohe'].get_feature_names_out(feature_cols_ohe))
except:
    ohe_names = [f'ohe_{i}' for i in range(preprocessor_fitted.named_transformers_['ohe'].transform(
        X_train[feature_cols_ohe].head(1)).shape[1])]

all_feature_names = num_names + bin_names + te_names + ohe_names
print(f'Total features after preprocessing: {len(all_feature_names)}')

In [ ]:
# 8.1 Logistic Regression coefficients
lr_model = results['Logistic Regression']['model']
lr_coefs = lr_model.named_steps['clf'].coef_[0]
lr_importance = pd.Series(np.abs(lr_coefs), index=all_feature_names[:len(lr_coefs)])
lr_top15 = lr_importance.nlargest(15)

# 8.2 Random Forest permutation importance
print('Computing permutation importance for Random Forest...')
X_test_transformed = results['Random Forest']['model'].named_steps['pre'].transform(X_test)
perm_imp = permutation_importance(
    results['Random Forest']['model'].named_steps['clf'],
    X_test_transformed, y_test,
    n_repeats=10, random_state=RANDOM_STATE, scoring='roc_auc', n_jobs=-1
)
rf_importance = pd.Series(perm_imp.importances_mean, index=all_feature_names[:X_test_transformed.shape[1]])
rf_top15 = rf_importance.nlargest(15)

# 8.3 XGBoost feature importance (gain-based)
xgb_model = results['XGBoost']['model'].named_steps['clf']
xgb_imp = xgb_model.feature_importances_
xgb_importance = pd.Series(xgb_imp, index=all_feature_names[:len(xgb_imp)])
xgb_top15 = xgb_importance.nlargest(15)

In [ ]:
# 8.4 Feature importance comparison plot
top_features = list(set(lr_top15.index) | set(rf_top15.index) | set(xgb_top15.index))
top_features = sorted(top_features, key=lambda f: xgb_importance.get(f, 0), reverse=True)[:15]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, (title, imp) in zip(axes, [
    ('Logistic Regression\n(|Coefficient|)', lr_importance),
    ('Random Forest\n(Permutation Importance)', rf_importance),
    ('XGBoost\n(Feature Importance)', xgb_importance)
]):
    top = imp.reindex(top_features).fillna(0).sort_values()
    ax.barh(range(len(top)), top.values, color='#3498db', edgecolor='black')
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(top.index, fontsize=8)
    ax.set_title(title)
    ax.set_xlabel('Importance')

plt.suptitle('Feature Importance Comparison (Top 15 Features)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/feature_importance.png', bbox_inches='tight')
plt.show()

In [ ]:
# 8.5 SHAP analysis for XGBoost
print('Computing SHAP values for XGBoost...')
X_test_xgb = results['XGBoost']['model'].named_steps['pre'].transform(X_test)

explainer = shap.TreeExplainer(results['XGBoost']['model'].named_steps['clf'])
shap_values = explainer.shap_values(X_test_xgb)

fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_xgb,
                  feature_names=all_feature_names[:X_test_xgb.shape[1]],
                  max_display=15, show=False)
plt.title('SHAP Feature Importance (XGBoost)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/shap_beeswarm.png', bbox_inches='tight')
plt.show()

---
## 9. Error Analysis

In [ ]:
# Use the best model (XGBoost) for error analysis
best_model_name = max(model_metrics, key=lambda n: model_metrics[n]['AUC-ROC'])
y_pred_best = model_metrics[best_model_name]['y_pred']
y_prob_best = results[best_model_name]['y_prob']

test_analysis = X_test.copy()
test_analysis['y_true'] = y_test
test_analysis['y_pred'] = y_pred_best
test_analysis['y_prob'] = y_prob_best
test_analysis['error_type'] = 'Correct'
test_analysis.loc[(test_analysis['y_true'] == 0) & (test_analysis['y_pred'] == 1), 'error_type'] = 'False Positive'
test_analysis.loc[(test_analysis['y_true'] == 1) & (test_analysis['y_pred'] == 0), 'error_type'] = 'False Negative'

print(f'Error analysis using {best_model_name}:')
print(test_analysis['error_type'].value_counts())
print()

# Error rates by NAICS sector
print('False Negative rate by NAICS 2-digit sector:')
violations = test_analysis[test_analysis['y_true'] == 1]
fn_by_naics = violations.groupby('naics_2digit').apply(
    lambda g: (g['y_pred'] == 0).mean()
).sort_values(ascending=False)
print(fn_by_naics.round(3))
print()

# Error rates by program count
print('False Negative rate by number of programs:')
fn_by_prog = violations.groupby('n_programs').apply(
    lambda g: (g['y_pred'] == 0).mean()
).sort_index()
print(fn_by_prog.round(3))

In [ ]:
# Characterize false negatives vs true positives
print('\nFalse Negatives vs True Positives — key feature differences:')
fn_mask = (test_analysis['y_true'] == 1) & (test_analysis['y_pred'] == 0)
tp_mask = (test_analysis['y_true'] == 1) & (test_analysis['y_pred'] == 1)

compare_cols = ['fac_n_nc_qtrs', 'fac_last_nc_distance', 'fac_violation_trend',
                'fac_inspection_count', 'n_programs']
comparison = pd.DataFrame({
    'False Negatives': test_analysis.loc[fn_mask, compare_cols].mean(),
    'True Positives': test_analysis.loc[tp_mask, compare_cols].mean()
})
print(comparison.round(3))
print()
print('Interpretation: False negatives tend to be facilities with fewer prior violations')
print('and less regulatory activity — they are harder to predict because they look like compliant facilities.')

---
## 10. Sensitivity Analysis: Leakage Check

In [ ]:
# Train XGBoost WITHOUT cumulative count features to check for soft leakage
cumulative_cols = ['fac_inspection_count', 'fac_informal_count', 'fac_formal_action_count',
                   'fac_total_penalties', 'fac_penalty_count',
                   'caa_evaluation_count', 'cwa_inspection_count', 'rcra_inspection_count']

feature_cols_numeric_clean = [c for c in feature_cols_numeric if c not in cumulative_cols]

preprocessor_clean = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), feature_cols_numeric_clean),
        ('bin', 'passthrough', feature_cols_binary),
        ('te', TargetEncoder(smooth='auto', random_state=RANDOM_STATE), feature_cols_te),
        ('ohe', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='infrequent_if_exist'),
         feature_cols_ohe),
    ],
    remainder='drop'
)

xgb_clean = Pipeline([
    ('pre', preprocessor_clean),
    ('clf', xgb.XGBClassifier(**best_xgb_params, scale_pos_weight=scale_pos_weight,
                              eval_metric='auc', tree_method='hist', random_state=RANDOM_STATE))
])
xgb_clean.fit(X_train, y_train)
prob_clean = xgb_clean.predict_proba(X_test)[:, 1]
auc_clean = roc_auc_score(y_test, prob_clean)
auc_full = roc_auc_score(y_test, results['XGBoost']['y_prob'])

print('Sensitivity Analysis: Effect of Cumulative Count Features')
print(f'  With cumulative counts:    AUC-ROC = {auc_full:.4f}')
print(f'  Without cumulative counts: AUC-ROC = {auc_clean:.4f}')
print(f'  Difference:                {auc_full - auc_clean:+.4f}')
if abs(auc_full - auc_clean) < 0.02:
    print('  -> Minimal impact: cumulative counts do not substantially inflate performance.')
else:
    print('  -> Notable impact: cumulative counts contribute meaningfully (possible soft leakage).')

---
## 11. Final Summary

In [ ]:
print('=' * 70)
print('FINAL RESULTS SUMMARY')
print('=' * 70)
print(f'\nDataset: {len(df):,} small manufacturing facilities')
print(f'Positive class (violation): {df["target"].mean():.1%}')
print(f'Train/test split: 80/20 stratified')
print(f'\n--- Model Performance on Test Set ---')
for name in results:
    m = model_metrics[name]
    print(f'\n{name}:')
    print(f'  AUC-ROC: {m["AUC-ROC"]:.4f} {m["AUC-ROC CI"]}')
    print(f'  AUC-PR:  {m["AUC-PR"]:.4f}')
    print(f'  F1:      {m["F1"]:.4f} (threshold={m["Threshold"]:.3f})')
    print(f'  Precision: {m["Precision"]:.4f}, Recall: {m["Recall"]:.4f}')

best = max(model_metrics, key=lambda n: model_metrics[n]['AUC-ROC'])
print(f'\nBest model: {best} (AUC-ROC = {model_metrics[best]["AUC-ROC"]:.4f})')

print('\n--- Class Imbalance Results ---')
print(imb_df.round(4).to_string())

print(f'\n--- Leakage Sensitivity ---')
print(f'AUC with cumulative counts: {auc_full:.4f}')
print(f'AUC without:               {auc_clean:.4f}')
print(f'Difference:                {auc_full - auc_clean:+.4f}')